In [2]:
import json
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [3]:
# 1. Select Processing Device (GPU or CPU)
def device_getter():
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


device = device_getter()


In [4]:

# 2. Image Transformations
# Real photos are colorful (3 channels) and vary in size. We resize them to 64x64 pixels.
transform = transforms.Compose(
    [
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),  # Normalizing RGB channels
    ]
)

In [5]:
# 3. Downloading a Small Starter Flower Dataset
# This downloads a dataset split into flower folders automatically
import os
import urllib.request
import zipfile

url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
import tarfile

if not os.path.exists("./flower_photos"):
    print("Downloading flowers dataset... (this may take a minute)")
    urllib.request.urlretrieve(url, "flowers.tgz")
    with tarfile.open("flowers.tgz", "r:gz") as tar:
        tar.extractall()

# We will look only at 3 classes for speed: 'daisy', 'dandelion', 'roses'
# PyTorch ImageFolder scans directories and assigns numeric IDs to each folder name
full_dataset = datasets.ImageFolder(root="./flower_photos", transform=transform)

# Filter dataset down to just 3 clean flower types for simplicity
targets = [
    full_dataset.class_to_idx[x] for x in ["daisy", "dandelion", "roses"]
]
indices = [
    i for i, (_, label) in enumerate(full_dataset.samples) if label in targets
]
flower_dataset = torch.utils.data.Subset(full_dataset, indices)

# Remap folder labels into index 0, 1, and 2
class_mapping = {
    full_dataset.class_to_idx["daisy"]: 0,
    full_dataset.class_to_idx["dandelion"]: 1,
    full_dataset.class_to_idx["roses"]: 2,
}
for i in range(len(flower_dataset.dataset.samples)):
    old_label = flower_dataset.dataset.samples[i][1]
    if old_label in class_mapping:
        flower_dataset.dataset.samples[i] = (
            flower_dataset.dataset.samples[i][0],
            class_mapping[old_label],
        )

# Split into train and test sets
train_size = int(0.8 * len(flower_dataset))
test_size = len(flower_dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(
    flower_dataset, [train_size, test_size]
)

train_loader = DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=32, shuffle=False)

In [6]:
# 4. Define Your Custom Flower Neural Network
class MyFlowerModel(nn.Module):

    def __init__(self):
        super().__init__()
        # Input has 3 channels (Red, Green, Blue) instead of MNIST's 1 channel (Black/White)
        self.conv_layer = nn.Conv2d(
            in_channels=3, out_channels=16, kernel_size=3, padding=1
        )
        self.relu = nn.ReLU()
        self.maxpool = nn.MaxPool2d(kernel_size=2)
        self.flatten = nn.Flatten()

        # Image size tracking: 64x64 pixels downscaled by maxpool (2x2) becomes 32x32 pixels.
        # 16 output channels * 32 * 32 = 16384 features
        self.fc1 = nn.Linear(16384, 128)
        self.fc2 = nn.Linear(128, 3)  # 3 output nodes (Daisy, Dandelion, Rose)

    def forward(self, x):
        x = self.maxpool(self.relu(self.conv_layer(x)))
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        return self.fc2(x)

In [7]:
model = MyFlowerModel().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [12]:
# 5. Training Loop
print("\n--- Starting Flower Training ---")
start_time = time.time()
for epoch in range(5):  # 5 epochs is enough for a basic test run
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(
        f"Epoch [{epoch+1}/5] complete. Average Loss: {running_loss/len(train_loader):.4f}"
    )

print(f"Training finished in: {time.time() - start_time:.2f} seconds\n")



--- Starting Flower Training ---
Epoch [1/5] complete. Average Loss: 0.3364
Epoch [2/5] complete. Average Loss: 0.2578
Epoch [3/5] complete. Average Loss: 0.1974
Epoch [4/5] complete. Average Loss: 0.1417
Epoch [5/5] complete. Average Loss: 0.1111
Training finished in: 65.76 seconds



In [13]:
# 1. Switch the model to evaluation/testing mode
model.eval()

# 2. Initialize counters for tracking correct guesses
correct = 0
total = 0

print("--- Evaluating Model on Test Data ---")

# 3. Turn off the gradient engine since we are testing, not training
with torch.no_grad():
    for images, labels in test_loader:
        # Move images and labels to your active processing device (GPU or CPU)
        images, labels = images.to(device), labels.to(device)

        # Run the images through the model to get raw score outputs
        outputs = model(images)

        # Pick the index with the highest score (this is the AI's final guess)
        _, predicted = torch.max(outputs.data, 1)

        # Count how many total images we have processed in this batch
        total += labels.size(0)

        # Compare the AI's guesses to the true labels and sum up the correct ones
        correct += (predicted == labels).sum().item()

# 4. Calculate and print the final percentage accuracy
final_accuracy = (correct / total) * 100
print(f"\nFinal Flower Test Accuracy: {final_accuracy:.2f}%")

--- Evaluating Model on Test Data ---

Final Flower Test Accuracy: 71.72%


In [10]:
# 1. Switch the model to evaluation mode
model.eval()

# 2. Create the dummy input matching the 3 color channels and 64x64 size
dummy_input = torch.randn(1, 3, 64, 64).to(device)

# 3. Export cleanly to a SINGLE standalone ONNX file
torch.onnx.export(
    model,
    dummy_input,
    "flower_model.onnx",
    input_names=["input"],
    output_names=["output"],
    export_params=True,
    dynamic_axes=None,  # Ensures simple fixed shape matching your web preprocessing
    external_data=False,  # CRITICAL: This forces everything into one file!
)
print("Successfully generated standalone 'flower_model.onnx'!")

[torch.onnx] Obtain model graph for `MyFlowerModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MyFlowerModel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Successfully generated standalone 'flower_model.onnx'!


C:\Users\Shreya\AppData\Local\Programs\Python\Python313\Lib\copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


In [11]:
# 7. Create the explicit labels JSON file manually
labels_dict = {"0": "Daisy", "1": "Dandelion", "2": "Rose"}
with open("labels.json", "w") as f:
    json.dump(labels_dict, f, indent=4)
print("Successfully generated 'labels.json'!")

Successfully generated 'labels.json'!
